In [14]:
import pandas as pd
import numpy as np
import datetime
import random
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import regularizers
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

In [18]:
# 1. SET RANDOM SEEDS FOR REPRODUCIBILITY
# This prevents the accuracy from jumping between 50% and 80% on every run
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

In [20]:
# 2. LOAD DATA
try:
    X_train = np.load('/kaggle/input/datasets/mudasarbhatti/financial-prediction-data/X_train.npy')
    X_test = np.load('/kaggle/input/datasets/mudasarbhatti/financial-prediction-data/X_test.npy')
    y_train = np.load('/kaggle/input/datasets/mudasarbhatti/financial-prediction-data/y_train.npy')
    y_test = np.load('/kaggle/input/datasets/mudasarbhatti/financial-prediction-data/y_test.npy')
    print(f"✓ Data loaded. Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")
except FileNotFoundError:
    print("Error: Check your dataset paths in Kaggle.")

✓ Data loaded. Training samples: 40, Test samples: 10


In [23]:
# 3. BUILD STABILIZED SIMPLERNN MODEL
model = Sequential([
    Input(shape=(5, 5)),
    SimpleRNN(32, 
              activation='tanh', 
              kernel_regularizer=regularizers.l2(0.01),
              recurrent_dropout=0.1),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [25]:
# 4. CONFIGURE CALLBACKS
# Increased patience to 25 to help the model escape local minima on small data
callbacks = [
    EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10)
]

In [26]:
# 5. TRAIN MODEL
print("\nStarting Reproducible Training...")
history = model.fit(
    X_train, y_train,
    epochs=300, # Increased max epochs; EarlyStopping will manage actual length
    batch_size=4,
    validation_split=0.2,
    class_weight={0: 1.4, 1: 0.6}, 
    callbacks=callbacks,
    verbose=1
)


Starting Reproducible Training...
Epoch 1/300
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 60ms/step - accuracy: 0.6440 - loss: 0.7887 - val_accuracy: 0.2500 - val_loss: 1.0212 - learning_rate: 0.0010
Epoch 2/300
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5390 - loss: 0.7948 - val_accuracy: 0.3750 - val_loss: 1.0008 - learning_rate: 0.0010
Epoch 3/300
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4960 - loss: 0.7628 - val_accuracy: 0.3750 - val_loss: 0.9643 - learning_rate: 0.0010
Epoch 4/300
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.5780 - loss: 0.6613 - val_accuracy: 0.3750 - val_loss: 0.9305 - learning_rate: 0.0010
Epoch 5/300
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5320 - loss: 0.7504 - val_accuracy: 0.6250 - val_loss: 0.9066 - learning_rate: 0.0010
Epoch 6/300
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6647 - loss: 0.7229 - val_accuracy: 0.6250 - val_loss: 0.8917 - learning_rate: 0.0010
Epoch 7/300
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5

In [28]:
# 6. EVALUATE ON TEST DATA
# The retracing warning from before is normal and can be ignored here
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype("int32")

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step


In [29]:
# 7. DISPLAY RESULTS
results = {
    'model': 'Reproducible_SimpleRNN',
    'accuracy': float(accuracy),
    'f1_score': float(f1),
    'precision': float(precision),
    'recall': float(recall),
    'confusion_matrix': cm.tolist(),
    'timestamp': datetime.datetime.now().isoformat()
}

results_df = pd.DataFrame([results])
print("\n" + "="*60)
print("FINAL EVALUATION RESULTS:")
print("="*60)
print(results_df.to_string(index=False))


FINAL EVALUATION RESULTS:
                 model  accuracy  f1_score  precision  recall confusion_matrix                  timestamp
Reproducible_SimpleRNN       0.8  0.666667        1.0     0.5 [[6, 0], [2, 2]] 2026-05-03T09:08:40.068733


In [31]:
# 8. SAVE OUTPUTS
# Using the modern .keras format to avoid legacy warnings
results_df.to_csv('rnn_stable_results.csv', index=False)
model.save('stable_rnn_model.keras')
print("\n✓ Model and results saved.")


✓ Model and results saved.
